## Make csv inbox volume

In [0]:

# ==========================================
# ENVIRONMENT CONFIGURATION — DEV
# ==========================================

CATALOG = "predictivemaintenance_dev"
SCHEMA = "raw_data"
VOLUME = "landing"

# Unity Catalog objects
landing_table = f"{CATALOG}.{SCHEMA}.landing_testdata"

# Volume paths
landing_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

csv_inbox = f"{landing_path}/csv_inbox"
schema_location = f"{landing_path}/_schemas/csv_autoloader"
checkpoint_location = f"{landing_path}/_checkpoints/csv_autoloader"

# Create volume if it does not exist
spark.sql(
    f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}"
)

# Create CSV inbox
dbutils.fs.mkdirs(csv_inbox)

print("Target catalog:", CATALOG)
print("Landing table:", landing_table)
print("CSV inbox:", csv_inbox)
print("Schema location:", schema_location)
print("Checkpoint location:", checkpoint_location)

display(dbutils.fs.ls(csv_inbox))


## Inspect the Landing Delta Table
This we we are going to finally drop the the csv too. But first we are going to put it in the csv_inbox

In [0]:

df_landing = spark.table(landing_table)

df_landing.printSchema()

display(df_landing.limit(5))

## Step 3 — Configure Auto Loader

### Read incoming CSV files

In [0]:

# Use the schema of our existing landing table
landing_schema = spark.table(landing_table).schema

# Configure Auto Loader
csv_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
    .schema(landing_schema)
    .load(csv_inbox)
    .select(*landing_schema.fieldNames())
)

### Write to the existing landing table

In [0]:
query = (
    csv_stream.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_location)
    .trigger(availableNow=True)
    .toTable(landing_table)
)

query.awaitTermination()

## Step 4 — Verify ingestion

In [0]:

from pyspark.sql import functions as F

# Find all CSV files currently in the inbox
csv_files = sorted([
    f.path
    for f in dbutils.fs.ls(csv_inbox)
    if f.name.lower().endswith(".csv")
])

if not csv_files:
    print("No CSV files found. Ingestion verification skipped.")

else:
    # Read the CSV files using the Landing table schema
    incoming_df = (
        spark.read
        .option("header", "true")
        .schema(landing_schema)
        .csv(csv_files)
    )

    landing_df = spark.table(landing_table)

    # Calculate incoming CSV statistics
    total_records = incoming_df.count()
    unique_udis = incoming_df.select("UDI").distinct().count()

    min_udi = incoming_df.agg(F.min("UDI")).first()[0]
    max_udi = incoming_df.agg(F.max("UDI")).first()[0]

    # Check whether every incoming UDI exists in Landing
    missing_records = (
        incoming_df.select("UDI").distinct()
        .join(
            landing_df.select("UDI").distinct(),
            on="UDI",
            how="left_anti"
        )
    )

    missing_count = missing_records.count()

    print("CSV files:", len(csv_files))
    print("Total incoming records:", total_records)
    print("Unique UDIs:", unique_udis)
    print("Minimum UDI:", min_udi)
    print("Maximum UDI:", max_udi)
    print("Records missing from Landing:", missing_count)

    # Integrity checks
    assert total_records == unique_udis, \
        "Duplicate UDIs detected in incoming CSV files"

    assert missing_count == 0, \
        "Some incoming records are missing from Landing"

    print("SUCCESS: CSV ingestion verification passed.")

    display(
        incoming_df.groupBy("Machine_failure").count()
    )


In [0]:
display(spark.table(landing_table))